# Temporal graph fraud detection for digital payments

Follow a payment from an account to its recipient. Test whether their recent activity improves fraud detection beyond the amount and time of the payment. The experiment uses synthetic transactions, including ordinary payments, legitimate group collections and payouts, and coordinated fraud episodes.

Run this notebook on Databricks Free Edition using a serverless Standard environment. The first code cell loads the included Python project into this notebook session. The configuration, training and results cells are below it.

In [ ]:
# This bundled wheel contains the inspectable Python sources in the project download.
import base64
import hashlib
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

wheel_bytes = base64.b64decode(
    "..."
)
if hashlib.sha256(wheel_bytes).hexdigest() != "483fa198e827addd1fcba3aa74eeaf955d59fd3bb1c63e1db519b4eba9e4746b":
    raise ValueError("The embedded project wheel did not match its build checksum.")
if "_payment_fraud_session" not in globals():
    _payment_fraud_session = TemporaryDirectory(prefix="payment-fraud-package-")
wheel_path = Path(_payment_fraud_session.name) / "payment_fraud-1.0.0-py3-none-any.whl"
wheel_path.write_bytes(wheel_bytes)
if str(wheel_path) not in sys.path:
    sys.path.insert(0, str(wheel_path))


In [ ]:
from importlib.metadata import version
import os
from pathlib import Path
from tempfile import TemporaryDirectory

# Serverless Free Edition uses Unity Catalog and restricts Spark configuration lookups.
os.environ["MLFLOW_REGISTRY_URI"] = "databricks-uc"

import mlflow
from databricks.sdk import WorkspaceClient
from IPython.display import Markdown, display as show_table

from payment_fraud.pipeline import run_pipeline

print(f"Project {version('payment-fraud')} | MLflow {mlflow.__version__}")

## Configure the experiment

The defaults generate 36,000 payments across 90 days. The first 20% of unique timestamps form the anomaly reference period, the next 40% train all three classifiers, the next 20% calibrate the alert thresholds, and the last 20% measure performance. Each threshold permits at most one false alert per 100 legitimate validation payments. The rate on later payments can differ.

In [ ]:
seed = 42
days = 90
transactions_per_day = 400
target_fpr = 0.01

workspace_user = spark.sql("SELECT current_user()").first()[0]
experiment_name = f"/Users/{workspace_user}/temporal-payment-fraud"

## Train and log three detectors

Compare a random forest using amount and time with a forest that also uses rolling graph features. The hybrid model adds an Isolation Forest anomaly score learned from legitimate payments in the reference period. MLflow records a parent run and three child runs in your workspace user folder.

In [ ]:
with TemporaryDirectory(prefix="payment-fraud-run-") as staging_directory:
    result = run_pipeline(
        output_dir=Path(staging_directory) / "results",
        seed=seed,
        days=days,
        transactions_per_day=transactions_per_day,
        target_fpr=target_fpr,
        tracking_uri="databricks",
        experiment_name=experiment_name,
    )

tracking = mlflow.tracking.MlflowClient(tracking_uri="databricks", registry_uri="databricks-uc")
run_info = tracking.get_run(result.run_id).info
print(f"{experiment_name}\nRun {run_info.run_id} | {run_info.status}")
print("MLflow artifacts:", ", ".join(entry.path for entry in tracking.list_artifacts(result.run_id)))
workspace_url = WorkspaceClient().config.host.rstrip("/")
run_url = f"{workspace_url}/#mlflow/experiments/{run_info.experiment_id}/runs/{run_info.run_id}"
show_table(Markdown(f"[Open this run in MLflow]({run_url})"))

## Inspect the holdout

Check the date boundaries and number of fraudulent payments in each period. Compare recall at the frozen threshold alongside the achieved false-positive rate. Precision measures the fraction of alerts that corresponded to fraud.

In [ ]:
show_table(result.evaluation.splits)
show_table(result.evaluation.metrics)

## Review coordinated activity

The pattern table counts the fraud payments detected and the rings with at least one detected payment. Below it, the 20 highest-scored hybrid alerts show which methods flagged each payment. Use the parent run's artifacts for the full holdout and report.

In [ ]:
show_table(result.evaluation.by_pattern)
predictions = result.evaluation.predictions
hybrid_alerts = predictions.loc[predictions["hybrid_alert"]]
review_columns = [
    "timestamp", "sender", "recipient", "amount", "pattern", "ring_id", "is_fraud",
    "baseline_alert", "graph_alert", "hybrid_score",
]
show_table(hybrid_alerts.nlargest(20, "hybrid_score")[review_columns])